# 01 · Datos base

**Pipeline del `main`** (`fichas/etapa_31_pipeline_main.md`). Los parámetros están solo en `parametros.py`; el código, en `funciones.py`.

Este paso:
1. arma el parquet base de 2019 desde el crudo, con la clase ternaria de la cátedra. La clase se calcula con 2019 entero y **se guarda solo hasta 201910**: 201911 y 201912 son del dashboard y quedan sin clase. **Si el parquet ya existe, no lo rehace: lo verifica** (solo rehace uno de una versión anterior, que guardaba la clase de 201911);
2. detecta los meses rotos (95 % o más de ceros o nulos en un mes aislado, más 6 a mano) y comprueba que son los 23 de siempre.

Corre una sola vez: todos los branches usan el mismo parquet.

In [ ]:
import importlib
import time

import numpy as np
import pandas as pd

import warnings

import parametros as P
import funciones as F

warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')   # falso positivo de pandas 2.2 con Python 3.14

importlib.reload(P)
importlib.reload(F)
P.RESULTADOS.mkdir(parents=True, exist_ok=True)
print('Corrida:', P.CORRIDA['nombre'], '·', len(P.CORRIDA['bloques']), 'bloques')

## 1. Parquet base

In [ ]:
t0 = time.time()
rehacer = not P.BASE_PARQUET.exists()
if not rehacer and F.clase_despues_del_tope(P):
    print(f'El parquet guarda la clase de meses posteriores a {P.MES_TOPE} (versión anterior): se rehace')
    rehacer = True
if rehacer:
    assert P.CRUDO.exists(), f'No encuentro el crudo en {P.CRUDO.resolve()}: ajustar CRUDO en parametros.py'
    F.construir_base(P)
    print(f'Parquet armado desde el crudo en {time.time() - t0:.0f} s')
por_mes = F.verificar_base(P)
print(f'Parquet OK: {sum(por_mes.values()):,} filas · filas y clases por mes como las esperadas · '
      f'sin clase después de {P.MES_TOPE}')

## 2. Meses rotos

Un mes roto se pasa a NA antes de calcular la historia, para que no contamine los lags de los meses siguientes.

In [ ]:
rotos = F.meses_rotos(P)
anterior = P.RESULTADOS_ANTERIORES / '01_meses_rotos.csv'
if anterior.exists():
    clave = lambda x: set(map(tuple, x[['variable', 'medida', 'foto_mes']].astype(str).to_numpy()))
    assert clave(rotos) == clave(pd.read_csv(anterior)), 'Los meses rotos no coinciden con los del 01'
rotos.to_csv(P.RESULTADOS / 'meses_rotos.csv', index=False)
print(f'Meses rotos: {len(rotos)} (más {len(P.ROTOS_EXTRA)} extra en parametros.py)')
rotos.pivot_table(index='foto_mes', values='variable', aggfunc='count').rename(columns={'variable': 'variables rotas'})